In [ ]:
# Colab/local bootstrap: mount Drive (if needed), clone/install package, init wandb.
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
REPO_DIR = pathlib.Path("/content/gradient-ascent")
DEFAULT_COLAB_OUT_DIR = pathlib.Path("/content/drive/MyDrive/gradient-ascent-out")

github_token = os.environ.get("GITHUB_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")
IN_COLAB = False

try:
    from google.colab import drive, userdata  # type: ignore

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)

    if github_token is None:
        github_token = userdata.get("GITHUB_TOKEN")
    if wandb_api_key is None:
        wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass


def _find_project_root_from_cwd() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return None


project_root = _find_project_root_from_cwd() or REPO_DIR
if not project_root.exists():
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
        subprocess.run(["git", "clone", clone_url], check=True)
    else:
        raise RuntimeError(
            "Repo checkout not found. Add GITHUB_TOKEN as env var/Colab secret, or clone manually."
        )

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {project_root}, but it was not found.")

repo_src = project_root / "src"
for path in [project_root, repo_src]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gradient_ascent.notebook_bootstrap import bootstrap_notebook_environment

bootstrap = bootstrap_notebook_environment(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    default_colab_out_dir=DEFAULT_COLAB_OUT_DIR,
    github_token=github_token,
    wandb_api_key=wandb_api_key,
)

project_root = bootstrap.project_root
IN_COLAB = bootstrap.in_colab
DEFAULT_OUT_DIR = bootstrap.default_out_dir
wandb = bootstrap.wandb

print(f"Changed working directory to {project_root}")
print(f"Default OUT_DIR: {DEFAULT_OUT_DIR}")


## Section 1 - Core training and unlearning checkpoints

This section trains the original and retrained models, then runs five unlearning baselines: gradient ascent, SSD, SalUn, certified removal, and SCRUB. For GA, the preset below stays faithful to vanilla gradient ascent on the forget set, but is made slightly stronger in an easy-to-justify way: a modestly higher learning rate, more forget-set updates per epoch, BatchNorm buffers still frozen for clean evaluation, and a looser clip so updates are visible without becoming unstable.

It also saves simple baseline-specific diagnostics that are easy to explain in a dissertation. For GA, these are the mean forget-set cross-entropy and gradient norm at each unlearning step. For SCRUB, they are the student-teacher KL divergence on the forget and retain sets, the retain-set cross-entropy, and retain/forget accuracy. Together these show whether SCRUB is separating from the teacher on forgotten data while still staying close on retained data.

In [ ]:
import warnings

from gradient_ascent.notebook_helpers import (
    prepare_notebook_runtime,
    run_and_display_notebook_core_pipeline,
)

warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_CLASSES = 10
OUT_DIR = DEFAULT_OUT_DIR
REUSE_EXISTING_CHECKPOINTS = True
REUSE_ORIGINAL_CHECKPOINT = True
REUSE_RETRAINED_CHECKPOINT = True  # Set False to force retraining the without-frog model only.
REUSE_UNLEARNED_CHECKPOINTS = True  # Reuse saved GA/SSD/SalUn/Certified/SCRUB outputs when available.
resnet_model_depth = 50

runtime = prepare_notebook_runtime(
    num_classes=NUM_CLASSES,
    out_dir=OUT_DIR,
    model_depth=resnet_model_depth,
)

device = runtime.device
USE_BF16 = runtime.use_bf16
trainset = runtime.trainset
testset = runtime.testset
use_cuda = runtime.use_cuda
num_workers = runtime.num_workers
model_factory = runtime.model_factory
core_config = runtime.core_config

print(f"Runtime prepared on device: {device}")

core_artifacts, wandb_run = run_and_display_notebook_core_pipeline(
    runtime,
    wandb_module=wandb,
    reuse_existing_checkpoints=REUSE_EXISTING_CHECKPOINTS,
    reuse_original_checkpoint=REUSE_ORIGINAL_CHECKPOINT,
    reuse_retrained_checkpoint=REUSE_RETRAINED_CHECKPOINT,
    reuse_unlearned_checkpoints=REUSE_UNLEARNED_CHECKPOINTS,
)


## Section 2 - Trajectories, MIA, and evolving similarity bar plots

Initialises shared similarity helpers, then runs snapshot trajectories for all five baselines (GA, SSD, SalUn, certified removal, and SCRUB), computes MIA trajectories, and generates PDF-friendly similarity summaries plus evolving bar plots for both references: unlearned-vs-retrained and unlearned-vs-original.

In [ ]:
# Unlearning algorithm comparison trajectories across all five baselines.
# Shared similarity setup is created here so this cell is self-contained.

from gradient_ascent.notebook_helpers import (
    prepare_similarity_setup,
    run_and_display_notebook_trajectory_pipeline,
)

similarity_setup = prepare_similarity_setup()
layer_names = similarity_setup.layer_names
metrics = similarity_setup.metrics
higher_better_metrics = similarity_setup.higher_better_metrics
lower_better_metrics = similarity_setup.lower_better_metrics
plot_metric_names = similarity_setup.plot_metric_names

trajectory_artifacts, trajectory_wandb_run = run_and_display_notebook_trajectory_pipeline(
    runtime,
    core_artifacts,
    similarity_setup,
    wandb_module=wandb,
)


## Section 3 - Combined cross-algorithm comparison

Overlays all five baselines, including SCRUB, in a single consolidated similarity + MIA comparison figure.

In [ ]:
# Integrated combined comparison figure: similarity + MIA trajectories.

from gradient_ascent.notebook_helpers import run_and_display_notebook_combined_comparison

combined_path = run_and_display_notebook_combined_comparison(runtime, wandb_module=wandb)
